# LangGraph Quickstart

LangGraphではLLMや各種ツールを内包したNode（エージェントのようなもの）の協調動作ワークフローをグラフ構造で定義します。

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END

def mock_llm(state: MessagesState):
    return {"messages": [{"role": "ai", "content": "hello world"}]}

graph = StateGraph(MessagesState)
graph.add_node(mock_llm)
graph.add_edge(START, "mock_llm")
graph.add_edge("mock_llm", END)
graph = graph.compile()

graph.invoke({"messages": [{"role": "user", "content": "hi!"}]})

## グラフ定義の基本的な流れ

LangGraphのワークフローを定義するグラフは、以下の構成要素からなります

- State: ワークフローの状態（各Nodeで更新されたメッセージが代表的）を保持する
- Node: 各種タスクを実行し、その結果に基づきStateを更新する
- Edge: Node間の遷移を定義する

### State

Stateは、保持したい状態をPython標準ライブラリの`typing_extensions.TypeDict`で定義します。
最低限、メッセージを保持するフィールドが必要となります。ここに`Annotated[list[AnyMessage], operator.add]`という型アノテーションをつけることで、新たなメッセージが付加的に保持されるようになります。

以下の例ではLLMの呼び出し回数を表す`llm_calls`というint型フィールドも追加しています。

In [ ]:
from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
import operator

class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    llm_calls: int


### Node

Nodeの種類として以下が代表的です。

- START: ワークフロー開始を表す特殊ノード
- Model: LLMでメッセージに基づく応答をするノード
- Tool: 自作処理や外部ツールを呼び出すノード
- END: ワークフロー終了を表す特殊ノード

Nodeは、基本的に関数で定義した処理を実装できます。

#### Model用Nodeの実装例

例えば、シンプルなModelノードに必要な処理は以下のように定義できます。注意点として、LLMだけでなくどのような条件でツールを呼び出すか判断するために、ツールの定義関数（判断のためのDockstringが重要）も`bind_tools`メソッドで一緒に渡す必要があります。

In [ ]:
from langchain.tools import tool
from langchain.chat_models import init_chat_model

# Define the model
model = init_chat_model(
    "claude-sonnet-4-6",
    temperature=0
)

# Define tools
@tool
def multiply(a: int, b: int) -> int:
    """Multiply `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a * b

@tool
def add(a: int, b: int) -> int:
    """Adds `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a + b

@tool
def divide(a: int, b: int) -> float:
    """Divide `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a / b

# Augment the LLM with tools
tools = [add, multiply, divide]
tools_by_name = {tool.name: tool for tool in tools}
model_with_tools = model.bind_tools(tools)

これらの処理を1つのNode用関数にまとめます。関数の戻り値で、Stateの各フィールドをどのように更新するかを定義します。
以下の例では、現在の"messages"フィールドをLLMに渡し、帰ってきた返答を"messages"フィールドに追加しています。また"llm_calls"フィールドは1追加しています。

In [ ]:
from langchain.messages import SystemMessage


def llm_call(state: dict):
    """LLM decides whether to call a tool or not"""

    return {
        "messages": [
            model_with_tools.invoke(
                [
                    SystemMessage(
                        content="You are a helpful assistant tasked with performing arithmetic on a set of inputs."
                    )
                ]
                + state["messages"]
            )
        ],
        "llm_calls": state.get('llm_calls', 0) + 1
    }

#### Tool呼び出し用ノードの実装例

以下の例では、直近のメッセージ（Modelノードが返したことを前提としている）にtool_calls指令（メッセージと`bind_tools`で渡したDocstringに基づきModelノードが指令を出すかを判断、tool_calls指令が含まれるかどうかは後述の条件付きエッジで判定）を走査し、ツールを実際に呼び出して結果をStateの"messages"フィールドに追加する処理をするNodeを実装しています

In [ ]:
from langchain.messages import ToolMessage


def tool_node(state: dict):
    """Performs the tool call"""

    result = []
    for tool_call in state["messages"][-1].tool_calls:
        tool = tools_by_name[tool_call["name"]]
        observation = tool.invoke(tool_call["args"])
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    return {"messages": result}

### Edge

EdgeはNode間の遷移を定義します。Edgeの実装は、遷移が条件付きかどうかで以下のように分かれます。

- 無条件に遷移するEdge: `add_edge`メソッドを使用
- 条件付きで遷移: 遷移条件を定義した関数を作成した上で`add_conditional_edges`メソッドを使用

例えば以下のケースでは、直近のメッセージ（Modelノードが返したことを前提としている）にtool_calls指令が含まれるかどうかを判定し、含まれる場合は"tool_node"に遷移、含まれない場合は"END"に遷移する処理を実装しています。

In [ ]:
from typing import Literal
from langgraph.graph import StateGraph, START, END


def should_continue(state: MessagesState) -> Literal["tool_node", END]:
    """Decide if we should continue the loop or stop based upon whether the LLM made a tool call"""

    messages = state["messages"]
    last_message = messages[-1]

    # If the LLM makes a tool call, then perform an action
    if last_message.tool_calls:
        return "tool_node"

    # Otherwise, we stop (reply to the user)
    return END

### グラフのビルド

ここまでで各State、Node、Edgeの内容が定義できたので、あとは一つのグラフに組み立てます。

In [ ]:
# Build workflow
agent_builder = StateGraph(MessagesState)

# Add nodes
agent_builder.add_node("llm_call", llm_call)
agent_builder.add_node("tool_node", tool_node)

# Add edges to connect nodes
agent_builder.add_edge(START, "llm_call")
agent_builder.add_conditional_edges(
    "llm_call",
    should_continue,
    ["tool_node", END]
)
agent_builder.add_edge("tool_node", "llm_call")

# Compile the agent
agent = agent_builder.compile()

# Show the agent
from IPython.display import Image, display
display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

# Invoke
from langchain.messages import HumanMessage
messages = [HumanMessage(content="Add 3 and 4.")]
messages = agent.invoke({"messages": messages})
for m in messages["messages"]:
    m.pretty_print()